<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/tb3py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Example of using TB3Py https://github.com/usnistgov/tb3py/ and get wannier90_hr.dat file.

In [ ]:
# Step 1: Download and extract Julia (change version if needed)
!wget https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.5-linux-x86_64.tar.gz
!tar -xvzf julia-1.11.5-linux-x86_64.tar.gz
!mv julia-1.11.5 /content/julia

# Step 2: Add Julia to current Python session PATH
import os
os.environ["PATH"] = "/content/julia/bin:" + os.environ["PATH"]

# Step 3: Check if Julia is available now
!which julia
!julia --version


In [ ]:
!julia -version

In [ ]:
!julia -e 'using Pkg; Pkg.add(["ThreeBodyTB", "Plots"]); Pkg.precompile()' &> /dev/null

In [ ]:
!pip install -q jarvis-tools spglib

In [ ]:
import os

pos="""Si2
1.0
3.3641499856336465 -2.5027128e-09 1.94229273881412
1.121382991333525 3.1717517190189715 1.9422927388141193
-2.5909987e-09 -1.8321133e-09 3.884586486670313
Si
2
Cartesian
3.92483875 2.77528125 6.7980237500000005
0.56069125 0.39646875 0.9711462500000001
"""
pos="""C2
1.0
1.2320971008984494 -2.1340542301747005 0.0
1.2320971008984494 2.1340542301747005 0.0
0.0 0.0 30.803073
C
2
Cartesian
0.0 0.0 1.9507584231000183
1.2321 0.7113514226999862 1.9507584231000183
"""
with open("POSCAR","w") as f:
    f.write(pos)

lines="""using ThreeBodyTB
c = makecrys("POSCAR")
energy, tbc, flag = scf_energy(c);
println("directgap, indirectgap, gaptype, bandwidth",string(ThreeBodyTB.BandStruct.band_summary(tbc)),tbc.efermi)
write_hr_dat(tbc,filename="my_hr.dat")
"""
with open("input.jl","w") as f:
     f.write(lines)
#cmd = "!julia input.jl"
#os.system(cmd)

In [ ]:
!julia input.jl

You can download my_hr.dat file by clicking on older icon on lef and download.

In [ ]:
!ls

In [ ]:
from jarvis.io.wannier.outputs import WannierHam
w = WannierHam(filename='my_hr.dat')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from jarvis.core.kpoints import generate_kgrid
kpoints = generate_kgrid([20, 20, 20])
energies, dos, pdos = w.dos(kpoints)
plt.plot(energies, dos)
plt.xlabel('E')
plt.ylabel('DOS')

In [ ]:
from jarvis.io.vasp.inputs import Poscar
atoms=Poscar.from_string(pos).atoms
filename='bands.png'
w.get_bandstructure_plot(atoms=atoms, filename=filename,yrange=[-8, 4])

In [ ]:
from IPython.display import Image
Image('bands.png')

See also https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ThreeBodyTB_julia.ipynb